## Check File Names

In [ ]:
# Execute shell command to list files in the HDFS Downloads directory.
import subprocess
result = subprocess.run(
    ["hdfs", "dfs", "-ls", "/user/ubuntu/Downloads"],
    capture_output=True, text=True
)
print(result.stdout)

Found 47 items
-rw-r--r--   3 ubuntu supergroup    4895650 2026-06-03 14:37 /user/ubuntu/Downloads/bus_station.csv
-rw-r--r--   3 ubuntu supergroup      20639 2026-06-03 14:38 /user/ubuntu/Downloads/drt_bus.csv
-rw-r--r--   3 ubuntu supergroup  342759992 2026-06-03 14:38 /user/ubuntu/Downloads/enterprise.csv
-rw-r--r--   3 ubuntu supergroup   29037064 2026-06-03 14:38 /user/ubuntu/Downloads/factory.csv
-rw-r--r--   3 ubuntu supergroup          5 2026-06-03 14:47 /user/ubuntu/Downloads/grid_daa_1K.cpg
-rw-r--r--   3 ubuntu supergroup      19634 2026-06-03 14:47 /user/ubuntu/Downloads/grid_daa_1K.dbf
-rw-r--r--   3 ubuntu supergroup        611 2026-06-03 14:47 /user/ubuntu/Downloads/grid_daa_1K.prj
-rw-r--r--   3 ubuntu supergroup     242044 2026-06-03 14:47 /user/ubuntu/Downloads/grid_daa_1K.shp
-rw-r--r--   3 ubuntu supergroup      14332 2026-06-03 14:47 /user/ubuntu/Downloads/grid_daa_1K.shx
-rw-r--r--   3 ubuntu supergroup          5 2026-06-03 14:47 /user/ubuntu/Downloads/grid_daba_

## Generate Gyeonggi-do Grids

In [ ]:
# Load multiple shapefiles into GeoPandas DataFrames and concatenate them into a single DataFrame.
import geopandas as gpd
import pandas as pd

files = [
    "grid_daa_1K",
    "grid_daba_1K",
    "grid_dasa_1K",
    "grid_laa_1K",
    "grid_laba_1K",
    "grid_lasa_1K",
]

base_path = "/mnt/c/Users/박나영/Desktop/hdfs/" 

gdfs = []
for f in files:
    gdf = gpd.read_file(f"{base_path}{f}.shp")
    gdfs.append(gdf)


merged = pd.concat(gdfs, ignore_index=True)
print(merged.columns.tolist()) 

['GRID_CD', 'geometry']


In [ ]:
# Display the first 5 rows of the merged grid DataFrame.
merged.head()

,GRID_CD,geometry
0,다아3700,"POLYGON ((937000 2000000, 937000 2001000, 9380..."
1,다아3800,"POLYGON ((938000 2000000, 938000 2001000, 9390..."
2,다아3900,"POLYGON ((939000 2000000, 939000 2001000, 9400..."
3,다아4000,"POLYGON ((940000 2000000, 940000 2001000, 9410..."
4,다아4001,"POLYGON ((940000 2001000, 940000 2002000, 9410..."


In [ ]:
# Fetch the geographical boundaries of South Korean provinces from an online JSON file.
import requests
import geopandas as gpd
import pandas as pd
from shapely.geometry import shape

url = "https://raw.githubusercontent.com/southkorea/southkorea-maps/master/kostat/2018/json/skorea-provinces-2018-geo.json"
resp = requests.get(url)
provinces = gpd.read_file(resp.text)

print(provinces.columns.tolist())

['name', 'base_year', 'name_eng', 'code', 'geometry']


In [ ]:
# Print the unique names of all provinces to verify the data.
print(provinces['name'].unique())

<ArrowStringArray>
[  '서울특별시',   '부산광역시',   '대구광역시',   '인천광역시',   '광주광역시',   '대전광역시',   '울산광역시',
 '세종특별자치시',     '경기도',     '강원도',    '충청북도',    '충청남도',    '전라북도',    '전라남도',
    '경상북도',    '경상남도', '제주특별자치도']
Length: 17, dtype: str


In [ ]:
# Filter the Gyeonggi-do polygon.
gyeonggi = provinces[provinces['name'] == '경기도']

# Align the CRS.
merged = merged.to_crs(gyeonggi.crs)

# Perform a spatial join to extract grids within Gyeonggi-do.
result = gpd.sjoin(merged, gyeonggi[['geometry']], how="inner", predicate="intersects")

print(result.shape)
print(result.columns.tolist())

(10847, 3)
['GRID_CD', 'geometry', 'index_right']


In [ ]:
# Check the results and the total number of grids within Gyeonggi-do.
result = result.drop(columns=['index_right'])
result = result.reset_index(drop=True)
print(len(result))
result.head()

10847

## Administrative & Welfare Center(행정복지센터) Data Preprocessing

In [ ]:
# Read the Administrative Welfare Center data from HDFS using PySpark.
new_village = spark.read.csv(
    "hdfs://ubuntu-master:9000/user/ubuntu/Downloads/new_village.csv",
    header=True,
    inferSchema=False,
    encoding="UTF-8"
)
new_village.show(5)
#new_village = new_village.toPandas()
#new_village.head(5)

+----+----+-------+-------------------+--------+--------------------------------+----------------+----------------+
|연번|시도| 시군구|             읍면동|우편번호|                            주소|        latitude|       longitude|
+----+----+-------+-------------------+--------+--------------------------------+----------------+----------------+
|1199|경기|김포시 |통진읍 행정복지센터|   10019| 경기도 김포시 통진읍 마송1로 77|37.6866731914427|126.600048757971|
|1200|경기|김포시 |고촌읍 행정복지센터|   10129|경기도 김포시 고촌읍 장차로 14  |37.6033046472851|126.770949050148|
|1201|경기|김포시 |양촌읍 행정복지센터|   10057|경기도 김포시 양촌읍 양곡1로6...|37.6572011427027|126.625431221903|
|1202|경기|김포시 |대곶면 행정복지센터|   10040| 경기 김포시 대곶면 율생로 83-23| 37.649110463461|126.581959491405|
|1203|경기|김포시 |월곶면 행정복지센터|   10024| 경기도 김포시 월곶면 군하로 263|37.7151376836215| 126.55284251058|
+----+----+-------+-------------------+--------+--------------------------------+----------------+----------------+
only showing top 5 rows


In [ ]:
# Define and apply a function to fetch coordinates using the Kakao API for missing latitude and longitude.
import requests
import pandas as pd
import time

KAKAO_API_KEY = "apiapi__"

def get_coordinates_kakao(address):
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    params = {"query": address}
    try:
        resp = requests.get(url, headers=headers, params=params, timeout=5)
        result = resp.json()
        if result["documents"]:
            lat = float(result["documents"][0]["y"])
            lon = float(result["documents"][0]["x"])
            return lat, lon
        return None, None
    except Exception as e:
        print(f"Error: {address} → {e}")
        return None, None

new_village[['latitude', 'longitude']] = new_village['주소'].apply(
    lambda x: pd.Series(get_coordinates_kakao(x))
)

print(new_village[new_village['latitude'].isna()])

In [ ]:
new_village.head(1)

In [ ]:
# Manually impute specific latitude and longitude values for rows where the API request failed.
new_village.loc[23, 'latitude']=37.65945
new_village.loc[23, 'longitude']=126.82959

new_village.loc[24, 'latitude']=37.6455092
new_village.loc[24, 'longitude']=126.8865

new_village.loc[452, 'latitude']=37.0226311
new_village.loc[452, 'longitude']=127.29049

new_village.loc[466, 'latitude']=37.0737881939662
new_village.loc[466, 'longitude']=127.028255085108

new_village.loc[471, 'latitude']=37.0405067
new_village.loc[471, 'longitude']=127.0504805

new_village.loc[519, 'latitude']=37.2068689831435
new_village.loc[519, 'longitude']=127.037279168976

In [ ]:
# Verify if there are any remaining missing values in the spatial coordinates.
new_village[new_village['latitude'].isna() | new_village['longitude'].isna()]

In [ ]:
new_village.head()

## Senior Center(경로당) Data Preprocessing

In [ ]:
# Read the Senior Center data from HDFS and convert the PySpark DataFrame to a Pandas DataFrame.
senior_center = spark.read.csv(
    "hdfs://ubuntu-master:9000/user/ubuntu/Downloads/senior_center.csv",
    header=True,
    inferSchema=False,
    encoding="UTF-8"
)
senior_center = senior_center.toPandas()
senior_center.head(5)

In [ ]:
# Rename the Korean coordinate column names to standard English 'latitude' and 'longitude'.
senior_center = senior_center.rename(columns={'WGS84위도': 'latitude'})
senior_center = senior_center.rename(columns={'WGS84경도': 'longitude'})

In [ ]:
# Drop rows that still contain missing spatial coordinates even after the API lookup process and reset the index.
senior_center = senior_center.dropna(subset=['latitude', 'longitude']).reset_index(drop=True)
senior_center.head(5)

In [ ]:
# Convert the latitude and longitude columns to float data types.
senior_center['latitude'] = senior_center['latitude'].astype(float)
senior_center['longitude'] = senior_center['longitude'].astype(float)

In [ ]:
senior_center.head()

## Medical Facility Data Preprocessing

In [ ]:
# Read the Medical Facility data from HDFS and convert it to a Pandas DataFrame.
medical = spark.read.csv(
    "hdfs://ubuntu-master:9000/user/ubuntu/Downloads/medical.csv",
    header=True,
    inferSchema=False,
    encoding="UTF-8"
)
medical = medical.toPandas()
medical.head(5)

In [ ]:
# Filter out facilities with a 'closed' business status and reset the index.
medical = medical[medical['영업상태명'] != '폐업'].reset_index(drop=True)
medical.head()

In [ ]:
# Check for missing values.
medical[medical['위도'].isna() | medical['경도'].isna()]

In [ ]:
# Rename the Korean coordinate column names to standard English 'latitude' and 'longitude'.
medical = medical.rename(columns={'위도': 'latitude'})
medical = medical.rename(columns={'경도': 'longitude'})

In [ ]:
# Convert the latitude and longitude columns to float data types.
medical['latitude'] = medical['latitude'].astype(float)
medical['longitude'] = medical['longitude'].astype(float)

# Manually fill in specific missing coordinates.
medical.loc[428, 'latitude']= 37.4983707318869
medical.loc[428, 'longitude']=126.7063879

medical.loc[700, 'latitude']=126.762109115188
medical.loc[700, 'longitude']=126.7063879

In [ ]:
print(medical[medical['latitude'].isna() | medical['latitude'].isna()])
medical.head()

## Pharmacy Data Preprocessing

In [ ]:
# Read the Pharmacy data from HDFS and convert it to a Pandas DataFrame.
pharmacy = spark.read.csv(
    "hdfs://ubuntu-master:9000/user/ubuntu/Downloads/pharmacy.csv",
    header=True,
    inferSchema=False,
    encoding="UTF-8"
)
pharmacy = pharmacy.toPandas()
pharmacy.head(5)

In [ ]:
# Rename coordinate columns to English
pharmacy = pharmacy.rename(columns={'WGS84위도': 'latitude'})
pharmacy = pharmacy.rename(columns={'WGS84경도': 'longitude'})

pharmacy[pharmacy['latitude'].isna() | pharmacy['latitude'].isna()]

In [ ]:
# Drop rows that still contain missing spatial coordinates even after the API lookup process and reset the index.
pharmacy = pharmacy.dropna(subset=['latitude', 'longitude']).reset_index(drop=True)
pharmacy.head()

## Market Data Preprocessing

In [ ]:
# Read the Market data from HDFS and convert it to a Pandas DataFrame.
market = spark.read.csv(
    "hdfs://ubuntu-master:9000/user/ubuntu/Downloads/market.csv",
    header=True,
    inferSchema=False,
    encoding="UTF-8"
)
market = market.toPandas()
market.head(5)

In [ ]:
# Rename coordinate columns to English
market = market.rename(columns={'WGS84위도': 'latitude'})
market = market.rename(columns={'WGS84경도': 'longitude'})

market[market['latitude'].isna() | market['latitude'].isna()]

In [ ]:
# Drop rows that still contain missing spatial coordinates even after the API lookup process and reset the index.
market = market.dropna(subset=['latitude', 'longitude']).reset_index(drop=True)
market.head()

## Restaurant(일반음식점) Data Preprocessing

In [ ]:
# Load the Restaurant data locally using Pandas to prevent Out-Of-Memory (OOM) errors in Spark.
import pandas as pd
import pandas as pd

restaurant = pd.read_csv("/mnt/c/Users/박나영/Desktop/hdfs/restaurant.csv", encoding="UTF-8")
print(restaurant.head())

In [ ]:
restaurant.head()

In [ ]:
# Remove closed restaurants and standardize the coordinate column names.
restaurant = restaurant[restaurant['통합영업상태명'] != '폐업'].reset_index(drop=True)
restaurant = restaurant[restaurant['영업상태명'] != '폐업'].reset_index(drop=True)

restaurant = restaurant.rename(columns={'위도': 'latitude'})
restaurant = restaurant.rename(columns={'경도': 'longitude'})

restaurant.head()

In [ ]:
restaurant[restaurant['latitude'].isna() | restaurant['latitude'].isna()]

In [ ]:
# Drop any restaurant records missing geographical coordinates to clean the dataset.
restaurant = restaurant.dropna(subset=['latitude', 'longitude']).reset_index(drop=True)
restaurant.head(1)

## Bus Station Data Preprocessing

In [ ]:
# Load the Bus Station data from the local file system using Pandas.
import pandas as pd

bus_station = pd.read_csv("/mnt/c/Users/박나영/Desktop/hdfs/bus_station.csv", encoding="UTF-8")
bus_station.head()

In [ ]:
# Rename coordinate columns and filter for rows with missing spatial data.
bus_station = bus_station.rename(columns={'WGS84위도': 'latitude'})
bus_station = bus_station.rename(columns={'WGS84경도': 'longitude'})

bus_station[bus_station['latitude'].isna() | bus_station['latitude'].isna()]

## Public Service Facility(공중이용시설) Data Preprocessing

In [ ]:
# Load the Public Service Facility data locally using Pandas.
import pandas as pd

service_facility = pd.read_csv("/mnt/c/Users/박나영/Desktop/hdfs/service_facility.csv", encoding="UTF-8")
service_facility.head()

In [ ]:
# Filter out facilities that are closed or out of business.
service_facility = service_facility[service_facility['영업상태명'] != '폐업 등'].reset_index(drop=True)

In [ ]:
# Standardize coordinate column names and drop records missing latitude or longitude.
service_facility = service_facility.rename(columns={'WGS84위도': 'latitude'})
service_facility = service_facility.rename(columns={'WGS84경도': 'longitude'})

service_facility[service_facility['latitude'].isna() | service_facility['latitude'].isna()]

In [ ]:
service_facility = service_facility.dropna(subset=['latitude', 'longitude']).reset_index(drop=True)
service_facility.head(1)

## Enterprise(사업체) Data Preprocessing

In [ ]:
# Load the Enterprise data locally using Pandas.
import pandas as pd

enterprise = pd.read_csv("/mnt/c/Users/박나영/Desktop/hdfs/enterprise.csv", encoding="UTF-8")
enterprise.head()

In [ ]:
# Rename the refined coordinate columns and remove rows with empty coordinate fields.
enterprise = enterprise.rename(columns={'정제WGS84위도': 'latitude'})
enterprise = enterprise.rename(columns={'정제WGS84경도': 'longitude'})

enterprise[enterprise['latitude'].isna() | enterprise['latitude'].isna()]

In [ ]:
enterprise = enterprise.dropna(subset=['latitude', 'longitude']).reset_index(drop=True)
enterprise.head(1)

## Factory Data Preprocessing

In [ ]:
# Read the Factory data from HDFS using PySpark and convert it to a Pandas DataFrame.
newnew_factory = spark.read.csv(
    "hdfs://ubuntu-master:9000/user/ubuntu/Downloads/newnew_factory.csv",
    header=True,
    inferSchema=False,
    encoding="UTF-8"
)
newnew_factory = newnew_factory.toPandas()
newnew_factory.head(5)

## Target Data (DRT Bus) Preprocessing

In [ ]:
# Load the target variable dataset (DRT bus operation data) locally.
import pandas as pd

drt_bus = pd.read_csv("/mnt/c/Users/박나영/Desktop/hdfs/drt_bus.csv", encoding="UTF-8")
drt_bus.head()

In [ ]:
# Binarize the target variable, setting operation counts of 1 or more to 1 (True) and 0 otherwise.
drt_bus['운행대수'] = (drt_bus['운행대수'] >= 1).astype(int)
print(drt_bus['운행대수'].value_counts())
drt_bus.head()